# T8: R01-R03横断 チーム/ドライバーパフォーマンストレンド分析

R01 オーストラリア、R02 中国、R03 日本の3GP横断で以下を分析:

1. チーム別ペーストレンド（燃料補正済）
2. ドライバー別成長率（正規化ペース %）
3. タイヤHARDデグレートのGP間比較
4. 一貫性スコア（標準偏差）
5. コンストラクターポイント累計推移

In [ ]:
import matplotlib
matplotlib.use("Agg")  # GUIなし環境対応
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import os, warnings
warnings.filterwarnings("ignore")

# パス設定
BASE_DIR = "/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised"
OUTPUT_DIR = os.path.join(BASE_DIR, "notebooks", "output")
DATA_DIR = os.path.join(BASE_DIR, "data")

print("パス設定完了")

In [ ]:
# ============================================================
# データ読み込み
# ============================================================
df_pace = pd.read_csv(os.path.join(OUTPUT_DIR, "fuel_corrected_pace.csv"))
df_deg  = pd.read_csv(os.path.join(OUTPUT_DIR, "deg_rates_clean.csv"))

# レースリザルト
race_dfs = {
    "R01": pd.read_csv(os.path.join(DATA_DIR, "2026_R01_Australia", "export", "race_results.csv")),
    "R02": pd.read_csv(os.path.join(DATA_DIR, "2026_R02_China",     "export", "race_results.csv")),
    "R03": pd.read_csv(os.path.join(DATA_DIR, "2026_R03_Japan",     "export", "race_results.csv")),
}

GP_KEYS = ["R01_Australia", "R02_China", "R03_Japan"]
print(f"fuel_corrected_pace: {len(df_pace)} 行")
print(f"deg_rates_clean: {len(df_deg)} 行")
df_pace.head()

In [ ]:
# チームカラー
TEAM_COLORS = {
    "McLaren":         "#F47600",
    "Ferrari":         "#ED1131",
    "Mercedes":        "#00D7B6",
    "Red Bull Racing": "#4781D7",
    "Aston Martin":    "#229971",
    "Williams":        "#1868DB",
    "Racing Bulls":    "#6C98FF",
    "Alpine":          "#00A1E8",
    "Haas F1 Team":    "#9C9FA2",
    "Audi":            "#F50537",
    "Cadillac":        "#909090",
}

STYLE = {
    "bg_color":   "#1a1a2e",
    "text_color": "#ffffff",
    "grid_color": "#333355",
}
print("設定完了")

## 1. チーム別ペーストレンド集計

In [ ]:
# チーム単位で2ドライバーの平均FuelCorrectedMedianPaceを計算
team_pace_records = []
for gp in GP_KEYS:
    gp_df = df_pace[df_pace["GP"] == gp]
    team_mean = gp_df.groupby("Team")["FuelCorrectedMedianPace"].mean().reset_index()
    team_mean["GP"] = gp
    team_pace_records.append(team_mean)

df_team_pace = pd.concat(team_pace_records, ignore_index=True)

# 正規化ペース計算
leader_pace = df_team_pace.groupby("GP")["FuelCorrectedMedianPace"].min().rename("LeaderPace")
df_team_pace = df_team_pace.join(leader_pace, on="GP")
df_team_pace["NormPace"] = (
    (df_team_pace["FuelCorrectedMedianPace"] - df_team_pace["LeaderPace"])
    / df_team_pace["LeaderPace"] * 100
)

df_team_pace.pivot_table(index="Team", columns="GP", values="NormPace").round(3)

## 2. ドライバー別正規化ペース（成長率）

In [ ]:
# ドライバー単位の正規化ペース計算
driver_pace_records = []
for gp in GP_KEYS:
    gp_df = df_pace[df_pace["GP"] == gp].copy()
    if gp_df.empty: continue
    leader_pace_val = gp_df["FuelCorrectedMedianPace"].min()
    gp_df["NormPace"] = (
        (gp_df["FuelCorrectedMedianPace"] - leader_pace_val)
        / leader_pace_val * 100
    )
    gp_df["GP_short"] = gp
    driver_pace_records.append(gp_df)

df_driver_norm = pd.concat(driver_pace_records, ignore_index=True)

# 一貫性スコア（3GP標準偏差）
driver_gp_norm = df_driver_norm.pivot_table(
    index="Driver", columns="GP_short", values="NormPace"
)
driver_gp_norm = driver_gp_norm.dropna(
    subset=["R01_Australia", "R02_China", "R03_Japan"], how="any"
)
driver_gp_norm["ConsistencyScore"] = driver_gp_norm[
    ["R01_Australia", "R02_China", "R03_Japan"]
].std(axis=1)

driver_team_map = df_driver_norm.groupby("Driver")["Team"].last()
driver_gp_norm["Team"] = driver_gp_norm.index.map(driver_team_map)

driver_gp_norm.sort_values("ConsistencyScore").round(3)

## 3. コンストラクターポイント累計

In [ ]:
# 各GPのコンストラクターポイント集計
constructor_points = {}
for key, rdf in race_dfs.items():
    team_pts = rdf.groupby("TeamName")["Points"].sum()
    constructor_points[key] = team_pts

all_teams = sorted(set().union(*[pts.index for pts in constructor_points.values()]))

# 累計計算
cumulative = {team: 0 for team in all_teams}
cumulative_by_gp = {}
for key in ["R01", "R02", "R03"]:
    pts = constructor_points.get(key, pd.Series(dtype=float))
    for team in all_teams:
        cumulative[team] += pts.get(team, 0)
    cumulative_by_gp[key] = cumulative.copy()

pd.DataFrame(cumulative_by_gp, index=all_teams).sort_values("R03", ascending=False)

## 4. グラフ生成（4パネル）

In [ ]:
# t8_run.py と同一ロジックでグラフを描画
# 詳細なグラフ描画はt8_run.pyを参照
import subprocess
result = subprocess.run(
    ["/usr/bin/python3", "/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised/notebooks/t8_run.py"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
else:
    print("グラフ生成完了")

## 5. CSV確認

In [ ]:
out_df = pd.read_csv("/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised/notebooks/output/cross_gp_team_trends.csv")
out_df.round(3)